# 🛡️ Guardian Deity — Women & Child Safety Detection Using Audio Signals

## 📌 Project Overview
This project aims to detect **danger situations** involving women and children using **audio signal classification**.  
By analyzing audio features extracted from `.wav` files, we train and compare multiple Machine Learning and Deep Learning models to classify sounds into three categories:

| Label | Description |
|-------|-------------|
| **Women** | Distress sounds from women |
| **Child** | Distress sounds from children |
| **Normal** | Background / safe sounds |

> 🎯 **Goal:** Identify the best performing model for real-time danger detection.

---

## 📂 Dataset
- **Name:** Audio Signal Dataset for Danger Detection of Women and Children  
- **Source:** Kaggle (Public Dataset)  
- **Format:** `.wav` audio files  
- **Classes:** Women, Child, Normal  
- **Split:** Train / Test (80% - 20%)

---

## 🧰 Libraries Used
`librosa` · `scikit-learn` · `tensorflow` · `pandas` · `numpy` · `matplotlib` · `seaborn`

## 🔗 Step 1: Mount Google Drive
Mount Google Drive to access the dataset stored in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 Step 2: Extract Dataset
Unzip the dataset from Google Drive to the Colab working directory.

> ⚠️ **Note:** Update the path below to match your own Google Drive folder location.

In [ ]:
!unzip '/content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip'

unzip:  cannot find or open /content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip, /content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip.zip or /content/drive/MyDrive/Audio Signal Dataset for Danger Detection of Women and Children.zip.ZIP.


## 🔊 Step 3: Load & Visualize a Sample Audio File
Before extracting features, we first load and visualize a sample audio file to understand the waveform structure.  
This helps confirm the dataset is loaded correctly and gives us a visual feel of the audio signal.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

# Load a sample audio file
audio_path = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train/Child/1002_0.wav'
y, sr = librosa.load(audio_path)

# Display the waveform
plt.figure(figsize=(14, 5))
librosa.display.waveshow(y, sr=sr)
plt.title('Waveform')
plt.show()

print(f'Audio loaded successfully. Sample rate: {sr} Hz')


/tmp/ipykernel_2215/2036166223.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train/Child/1002_0.wav'

## 📋 Step 4: Prepare Dataset Labels
Scan all audio files from each class folder (Women, Child, Normal) and create a structured DataFrame with:
- `file_path` → path to each `.wav` file  
- `label` → corresponding class name  

This label sheet is saved as an Excel file for reference.

In [ ]:
import os
import pandas as pd

# Define the root directory where your folders are stored
root_dir = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train'  # <-- replace this with your actual path

# Define your classes
categories = ['Women', 'Child', 'Normal']

# Prepare a list to collect data
data = []

# Loop through each category folder
for category in categories:
    folder_path = os.path.join(root_dir, category)
    for filename in os.listdir(folder_path):
        if filename.endswith('.wav'):
            file_path = os.path.join(folder_path, filename)
            data.append({'file_path': file_path, 'label': category})

# Create a DataFrame
df = pd.DataFrame(data)

# Save to Excel
output_path = os.path.join(root_dir, 'audio_dataset_labels.xlsx')
df.to_excel(output_path, index=False)

print(f"Excel sheet saved at: {output_path}")

## 🎵 Step 5: Audio Feature Extraction
We extract the following audio features from each `.wav` file using the `librosa` library:

| Feature | Description |
|---------|-------------|
| **ZCR** (Zero Crossing Rate) | How often the signal crosses zero — captures noisiness |
| **RMS** (Root Mean Square Energy) | Average energy/loudness of the audio |
| **Spectral Centroid** | "Center of mass" of the spectrum — captures brightness |
| **MFCC 1–13** | Mel Frequency Cepstral Coefficients — captures tonal patterns |

These 16 features form the input to all our models.

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

# Define your dataset directory
root_dir = '/content/Audio Signal Dataset for Danger Detection of Women┬аand┬аChildren/train'  # <- replace with your actual path
categories = ['Women', 'Child', 'Normal']

# List to store the final data
data = []

# Feature extraction function
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    features = {
        "zcr": np.mean(librosa.feature.zero_crossing_rate(y)),
        "rms": np.mean(librosa.feature.rms(y=y)),
        "spectral_centroid": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
    }

    # MFCCs: We take mean of first 13 coefficients
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}'] = np.mean(mfccs[i])

    return features

# Loop through each category and each audio file
for label in categories:
    folder_path = os.path.join(root_dir, label)
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            try:
                features = extract_features(file_path)
                features['label'] = label
                data.append(features)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

# Create DataFrame
df = pd.DataFrame(data)

# Save to Excel (optional)
df.to_excel(os.path.join(root_dir, 'audio_features_dataset.xlsx'), index=False)

print("Feature extraction complete. Excel file saved.")

## ✂️ Step 6: Train-Test Split
Split the extracted features into training and testing sets using an **80/20 ratio**.  
- `X` → feature columns (ZCR, RMS, Spectral Centroid, MFCC 1–13)  
- `y` → target label (Women / Child / Normal)

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Separate features (X) and labels (y)
# Define the input features by excluding the 'label' column
inputs = [col for col in df.columns if col != 'label']
X = df[inputs]
y = df['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 🌲 Step 7: Model 1 — Random Forest Classifier
Train a **Random Forest** model with 100 decision trees.  
Random Forest is an ensemble method that combines multiple decision trees to improve accuracy and reduce overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

## 📊 Step 8: Model 2 & 3 — Logistic Regression & Gaussian Naive Bayes
Train and evaluate two more classical ML models:
- **Logistic Regression** — a linear model that predicts class probabilities
- **Gaussian Naive Bayes** — a probabilistic model based on Bayes' theorem, assumes feature independence

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
log_reg_model.fit(X_train, y_train)

# Predict and evaluate Logistic Regression
y_pred_lr = log_reg_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy of the Logistic Regression model: {accuracy_lr:.4f}")
print("\nClassification Report (Logistic Regression):\n", classification_report(y_test, y_pred_lr))

print("-" * 30) # Separator for clarity

# Initialize and train the Gaussian Naive Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict and evaluate Naive Bayes
y_pred_nb = nb_model.predict(X_test)
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"Accuracy of the Gaussian Naive Bayes model: {accuracy_nb:.4f}")
print("\nClassification Report (Gaussian Naive Bayes):\n", classification_report(y_test, y_pred_nb))

## 📉 Step 9: Confusion Matrix Visualization
A **Confusion Matrix** shows how well each model predicted each class.  
- Rows = Actual labels  
- Columns = Predicted labels  
- Diagonal = Correct predictions ✅

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix for Random Forest
y_pred = model.predict(X_test)
cm_rf = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Generate confusion matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', xticklabels=log_reg_model.classes_, yticklabels=log_reg_model.classes_)
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Generate confusion matrix for Gaussian Naive Bayes
cm_nb = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues', xticklabels=nb_model.classes_, yticklabels=nb_model.classes_)
plt.title('Confusion Matrix - Gaussian Naive Bayes')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 🧠 Step 10: Model 4 — TCN with Attention (Deep Learning)
We build a custom **Temporal Convolutional Network (TCN)** with an **Attention mechanism** using TensorFlow/Keras.

### Why TCN?
- TCN uses **dilated causal convolutions** to capture long-range temporal patterns in audio
- Each TCN block has a **residual connection** to prevent vanishing gradients

### Why Attention?
- The **Attention layer** allows the model to focus on the most important time steps in the audio sequence
- This improves classification accuracy especially for short distress sounds

### Architecture:
`Input → TCN Blocks (x4) → Attention Layer → Dense (Softmax) → Output`

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Reshape, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv1D, Activation, Add, Multiply
from tensorflow.keras import backend as K
from tensorflow.keras.initializers import RandomNormal
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import Layer

# Define the TCN block
def tcn_block(input_layer, filters, kernel_size, dilation_rate, dropout_rate, activation='relu'):
    # Save the input for the residual connection
    residual = input_layer

    # First convolutional layer
    conv1 = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal',
                   kernel_initializer=RandomNormal(mean=0.0, stddev=0.02))(input_layer)
    conv1 = Activation(activation)(conv1)
    conv1 = Dropout(dropout_rate)(conv1)

    # Second convolutional layer
    conv2 = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal',
                   kernel_initializer=RandomNormal(mean=0.0, stddev=0.02))(conv1)
    conv2 = Activation(activation)(conv2)
    conv2 = Dropout(dropout_rate)(conv2)

    # Residual connection
    # If the number of filters changed, we need a 1x1 convolution for the residual
    if K.int_shape(residual)[-1] != filters:
        residual = Conv1D(filters, 1, padding='same')(residual)

    # Add the residual to the output of the block
    output_layer = Add()([residual, conv2])
    output_layer = Activation(activation)(output_layer) # Activation after adding residual

    return output_layer

# Custom Attention Layer
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape is (None, sequence_length, features)
        self.W = self.add_weight(name='attention_w', shape=(input_shape[-1], input_shape[-1]),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='attention_b', shape=(input_shape[-1],),
                                 initializer='zeros', trainable=True)
        # u should have the same dimension as features, but will be used as a column vector
        self.u = self.add_weight(name='attention_u', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        # x shape is (None, sequence_length, features)
        # uit shape will be (None, sequence_length, features)
        uit = K.tanh(K.dot(x, self.W) + self.b)

        # Compute the attention weights
        # ait shape will be (None, sequence_length, 1) after dot product with u
        ait = K.dot(uit, self.u)
        # Remove the last dimension to get shape (None, sequence_length)
        ait = K.squeeze(ait, axis=-1)

        a = K.exp(ait)

        # Apply softmax to get attention probabilities
        a /= K.cast(K.sum(a, axis=1, keepdims=True) + K.epsilon(), K.floatx())

        # Apply the attention weights to the input
        # weighted_input shape will be (None, sequence_length, features)
        weighted_input = x * K.expand_dims(a, axis=-1)

        # Sum the weighted input along the time dimension to get the context vector
        # result shape will be (None, features)
        result = K.sum(weighted_input, axis=1)

        return result

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])


# Define the TCN model with Attention
def build_tcn_model_with_attention(input_shape, num_classes, num_channels, kernel_size, dropout_rate, num_blocks):
    input_layer = Input(shape=input_shape)
    x = input_layer

    # Add TCN blocks
    for i in range(num_blocks):
        dilation_size = 2 ** i
        x = tcn_block(x, num_channels, kernel_size, dilation_rate=dilation_size, dropout_rate=dropout_rate)

    # Add the Attention layer
    x = Attention()(x)


    # Dense layers for classification
    output_layer = Dense(num_classes, activation='softmax')(x)

    model = tf.keras.models.Model(inputs=input_layer, outputs=output_layer)
    return model

# Model parameters
input_shape = (X_train.shape[1], 1) # Input shape for TCN: (sequence_length, features)
num_classes = 3 # Women, Child, Normal
num_channels = 64 # Number of filters in convolutional layers
kernel_size = 2 # Kernel size for convolutional layers
dropout_rate = 0.2
num_blocks = 4 # Number of TCN blocks

# Reshape input data for TCN (add a channel dimension)
# Ensure X_train and X_test are pandas DataFrames or NumPy arrays before reshaping
if isinstance(X_train, pd.DataFrame):
    X_train_tcn = X_train.values.reshape(X_train.shape[0], X_train.shape[1], 1)
else:
    X_train_tcn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

if isinstance(X_test, pd.DataFrame):
    X_test_tcn = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)
else:
     X_test_tcn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Encode the string labels to integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


# Build the TCN model with Attention
tcn_attention_model = build_tcn_model_with_attention(input_shape, num_classes, num_channels, kernel_size, dropout_rate, num_blocks)

# Compile the model
tcn_attention_model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Print the model summary
tcn_attention_model.summary()

# Train the model
print("\nTraining the TCN model with Attention...")
history_tcn_attention = tcn_attention_model.fit(X_train_tcn, y_train_encoded, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model
loss_tcn_attention, accuracy_tcn_attention = tcn_attention_model.evaluate(X_test_tcn, y_test_encoded)

print(f"\nTest Loss (TCN with Attention): {loss_tcn_attention:.4f}")
print(f"Test Accuracy (TCN with Attention): {accuracy_tcn_attention:.4f}")

## 📈 Step 11: TCN Training History & Accuracy Comparison
Visualize how the TCN model learned over 50 epochs:
- **Training vs Validation Accuracy** — checks if model is learning well
- **Training vs Validation Loss** — checks if model is overfitting  

Also compare overall accuracy of all 4 models using a bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot TCN training history
plt.figure(figsize=(12, 6))
plt.plot(history_tcn_attention.history['accuracy'], label='Training Accuracy')
plt.plot(history_tcn_attention.history['val_accuracy'], label='Validation Accuracy')
plt.title('TCN Model Training History (Accuracy)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history_tcn_attention.history['loss'], label='Training Loss')
plt.plot(history_tcn_attention.history['val_loss'], label='Validation Loss')
plt.title('TCN Model Training History (Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Create a bar chart comparing model accuracies
model_names = ['Random Forest', 'Logistic Regression', 'Gaussian Naive Bayes', 'TCN with Attention']
accuracies = [model.score(X_test, y_test), accuracy_lr, accuracy_nb, accuracy_tcn_attention] # Use model.score for RF accuracy

plt.figure(figsize=(10, 6))
sns.barplot(x=model_names, y=accuracies, palette='viridis')
plt.title('Comparison of Model Accuracies')
plt.ylabel('Accuracy')
plt.ylim(0, 1) # Accuracy is between 0 and 1
plt.show()

## 🌳 Step 12: Model 5 — Decision Tree Classifier
Train a **Decision Tree** model as an additional baseline comparison.  
Decision Trees are easy to interpret and explain — useful for understanding feature importance.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree Classifier
decision_tree_model = DecisionTreeClassifier(random_state=42)

# Train the model on the training data
decision_tree_model.fit(X_train, y_train)

print("Decision Tree model trained successfully.")

## 🏆 Step 13: Final Accuracy Summary
Print the final accuracy of all 3 key models side by side for quick comparison.

In [ ]:
print(f"Random Forest Model Accuracy: {model.score(X_test, y_test):.4f}")
print(f"TCN with Attention Model Accuracy: {accuracy_tcn_attention:.4f}")
print(f"Decision Tree Model Accuracy: {decision_tree_model.score(X_test, y_test):.4f}")

## 📊 Step 14: Detailed Metrics — Precision, Recall & F1-Score
Compare all 5 models across all 3 classes using:
- **Precision** — out of all predicted positives, how many were actually positive?
- **Recall** — out of all actual positives, how many did we correctly predict?
- **F1-Score** — harmonic mean of Precision and Recall (best overall metric)

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Import numpy for radar plot

# Get classification reports for each model
report_rf = classification_report(y_test, model.predict(X_test), output_dict=True)
report_lr = classification_report(y_test, y_pred_lr, output_dict=True)
report_nb = classification_report(y_test, y_pred_nb, output_dict=True)

# For TCN, we need to get predictions and then the report
y_pred_tcn_encoded = tcn_attention_model.predict(X_test_tcn).argmax(axis=1)
y_pred_tcn = label_encoder.inverse_transform(y_pred_tcn_encoded)
report_tcn = classification_report(y_test, y_pred_tcn, output_dict=True)


# Extract metrics for each class and model
metrics_data = []
for class_name in ['Women', 'Child', 'Normal']:
    metrics_data.append({
        'Model': 'Random Forest',
        'Class': class_name,
        'Metric': 'Precision',
        'Score': report_rf[class_name]['precision']
    })
    metrics_data.append({
        'Model': 'Random Forest',
        'Class': class_name,
        'Metric': 'Recall',
        'Score': report_rf[class_name]['recall']
    })
    metrics_data.append({
        'Model': 'Random Forest',
        'Class': class_name,
        'Metric': 'F1-score',
        'Score': report_rf[class_name]['f1-score']
    })

    metrics_data.append({
        'Model': 'Logistic Regression',
        'Class': class_name,
        'Metric': 'Precision',
        'Score': report_lr[class_name]['precision']
    })
    metrics_data.append({
        'Model': 'Logistic Regression',
        'Class': class_name,
        'Metric': 'Recall',
        'Score': report_lr[class_name]['recall']
    })
    metrics_data.append({
        'Model': 'Logistic Regression',
        'Class': class_name,
        'Metric': 'F1-score',
        'Score': report_lr[class_name]['f1-score']
    })

    metrics_data.append({
        'Model': 'Gaussian Naive Bayes',
        'Class': class_name,
        'Metric': 'Precision',
        'Score': report_nb[class_name]['precision']
    })
    metrics_data.append({
        'Model': 'Gaussian Naive Bayes',
        'Class': class_name,
        'Metric': 'Recall',
        'Score': report_nb[class_name]['recall']
    })
    metrics_data.append({
        'Model': 'Gaussian Naive Bayes',
        'Class': class_name,
        'Metric': 'F1-score',
        'Score': report_nb[class_name]['f1-score']
    })

    metrics_data.append({
        'Model': 'TCN with Attention',
        'Class': class_name,
        'Metric': 'Precision',
        'Score': report_tcn[class_name]['precision']
    })
    metrics_data.append({
        'Model': 'TCN with Attention',
        'Class': class_name,
        'Metric': 'Recall',
        'Score': report_tcn[class_name]['recall']
    })
    metrics_data.append({
        'Model': 'TCN with Attention',
        'Class': class_name,
        'Metric': 'F1-score',
        'Score': report_tcn[class_name]['f1-score']
    })


metrics_df = pd.DataFrame(metrics_data)

# Plotting
plt.figure(figsize=(15, 8))
sns.barplot(x='Class', y='Score', hue='Model', data=metrics_df, palette='viridis', errorbar=None)
plt.title('Comparison of Model Performance by Class and Metric')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.show()

# You can also plot each metric separately for better clarity if needed
metrics_to_plot = ['Precision', 'Recall', 'F1-score']
for metric in metrics_to_plot:
    plt.figure(figsize=(15, 8))
    sns.barplot(x='Class', y='Score', hue='Model', data=metrics_df[metrics_df['Metric'] == metric], palette='viridis', errorbar=None)
    plt.title(f'Comparison of {metric} by Class and Model')
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.show()

# --- Start of code moved from cell nYEp9edrb0Hq ---

# Filter the DataFrame to include only the metrics for the radar plot
radar_data = metrics_df[metrics_df['Metric'].isin(['Precision', 'Recall', 'F1-score'])].copy()

# Pivot the data to have models as columns and metrics/classes as rows
radar_pivot = radar_data.pivot_table(index=['Class', 'Metric'], columns='Model', values='Score')

# Reset index to make Class and Metric columns
radar_pivot = radar_pivot.reset_index()

# Create a combined label for the radar plot
radar_pivot['Label'] = radar_pivot['Class'] + ' - ' + radar_pivot['Metric']

# Set the combined label as the index
radar_pivot = radar_pivot.set_index('Label')

# Drop the original Class and Metric columns
radar_pivot = radar_pivot.drop(columns=['Class', 'Metric'])

# --- Create the Radar Plot ---

labels = np.array(radar_pivot.index)
num_vars = len(labels)

# Calculate angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# Complete the loop for plotting
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Plot data for each model
for model_name in radar_pivot.columns:
    values = radar_pivot[model_name].tolist()
    values += values[:1] # Complete the loop
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name)

# Add labels and title
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_title('Model Performance Comparison (Radar Plot)', size=15, color='black', y=1.1)
ax.grid(True)
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

plt.show()

# --- End of code moved from cell nYEp9edrb0Hq ---

### 🎯 Precision Scores by Model & Class

In [ ]:
from sklearn.metrics import classification_report

# Get classification reports for each model
report_rf = classification_report(y_test, model.predict(X_test), output_dict=True)
report_dt = classification_report(y_test, decision_tree_model.predict(X_test), output_dict=True)
report_lr = classification_report(y_test, log_reg_model.predict(X_test), output_dict=True)
report_nb = classification_report(y_test, nb_model.predict(X_test), output_dict=True)


# For TCN with Attention, we need to get predictions and then the report
y_pred_tcn_encoded = tcn_attention_model.predict(X_test_tcn).argmax(axis=1)
y_pred_tcn = label_encoder.inverse_transform(y_pred_tcn_encoded)
report_tcn_attention = classification_report(y_test, y_pred_tcn, output_dict=True)

print("Precision Scores:")
print("-" * 20)

# Print precision for each class for Random Forest
print("Random Forest:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_rf[class_name]['precision']:.4f}")
print("-" * 20)

# Print precision for each class for Decision Tree
print("Decision Tree:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_dt[class_name]['precision']:.4f}")
print("-" * 20)

# Print precision for each class for TCN with Attention
print("TCN with Attention:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_tcn_attention[class_name]['precision']:.4f}")
print("-" * 20)

# Print precision for each class for Logistic Regression
print("Logistic Regression:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_lr[class_name]['precision']:.4f}")
print("-" * 20)

# Print precision for each class for Gaussian Naive Bayes
print("Gaussian Naive Bayes:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_nb[class_name]['precision']:.4f}")
print("-" * 20)

### 🔁 Recall Scores by Model & Class

In [ ]:
from sklearn.metrics import classification_report

# Get classification reports for each model
report_rf = classification_report(y_test, model.predict(X_test), output_dict=True)
report_dt = classification_report(y_test, decision_tree_model.predict(X_test), output_dict=True)
report_lr = classification_report(y_test, log_reg_model.predict(X_test), output_dict=True)
report_nb = classification_report(y_test, nb_model.predict(X_test), output_dict=True)

# For TCN with Attention, we need to get predictions and then the report
y_pred_tcn_encoded = tcn_attention_model.predict(X_test_tcn).argmax(axis=1)
y_pred_tcn = label_encoder.inverse_transform(y_pred_tcn_encoded)
report_tcn_attention = classification_report(y_test, y_pred_tcn, output_dict=True)

print("Recall Scores:")
print("-" * 20)

# Print recall for each class for Random Forest
print("Random Forest:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_rf[class_name]['recall']:.4f}")
print("-" * 20)

# Print recall for each class for Decision Tree
print("Decision Tree:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_dt[class_name]['recall']:.4f}")
print("-" * 20)

# Print recall for each class for TCN with Attention
print("TCN with Attention:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_tcn_attention[class_name]['recall']:.4f}")
print("-" * 20)

# Print recall for each class for Logistic Regression
print("Logistic Regression:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_lr[class_name]['recall']:.4f}")
print("-" * 20)

# Print recall for each class for Gaussian Naive Bayes
print("Gaussian Naive Bayes:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_nb[class_name]['recall']:.4f}")
print("-" * 20)

### ⚖️ F1-Scores by Model & Class

In [ ]:
from sklearn.metrics import classification_report

# Get classification reports for each model
report_rf = classification_report(y_test, model.predict(X_test), output_dict=True)
report_dt = classification_report(y_test, decision_tree_model.predict(X_test), output_dict=True)
report_lr = classification_report(y_test, log_reg_model.predict(X_test), output_dict=True)
report_nb = classification_report(y_test, nb_model.predict(X_test), output_dict=True)

# For TCN with Attention, we need to get predictions and then the report
y_pred_tcn_encoded = tcn_attention_model.predict(X_test_tcn).argmax(axis=1)
y_pred_tcn = label_encoder.inverse_transform(y_pred_tcn_encoded)
report_tcn_attention = classification_report(y_test, y_pred_tcn, output_dict=True)

print("F1-Scores:")
print("-" * 20)

# Print F1-score for each class for Random Forest
print("Random Forest:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_rf[class_name]['f1-score']:.4f}")
print("-" * 20)

# Print F1-score for each class for Decision Tree
print("Decision Tree:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_dt[class_name]['f1-score']:.4f}")
print("-" * 20)

# Print F1-score for each class for TCN with Attention
print("TCN with Attention:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_tcn_attention[class_name]['f1-score']:.4f}")
print("-" * 20)

# Print F1-score for each class for Logistic Regression
print("Logistic Regression:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_lr[class_name]['f1-score']:.4f}")
print("-" * 20)

# Print F1-score for each class for Gaussian Naive Bayes
print("Gaussian Naive Bayes:")
for class_name in ['Women', 'Child', 'Normal']:
    print(f"  {class_name}: {report_nb[class_name]['f1-score']:.4f}")
print("-" * 20)

## 📐 Step 15: AUC Score (Area Under the ROC Curve)
**AUC** measures the model's ability to distinguish between classes.  
- AUC = 1.0 → Perfect model  
- AUC = 0.5 → Random guessing  
We use **One-vs-Rest** strategy for multi-class AUC calculation.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelBinarizer

print("AUC Scores (One-vs-Rest):")
print("-" * 30)

# Label Binarizer for one-vs-rest AUC calculation
lb = LabelBinarizer()
y_test_binarized = lb.fit_transform(y_test)

# Function to calculate AUC for a given model
def calculate_auc(model, X_test, y_test_binarized, label_binarizer):
    # For scikit-learn models, we can use predict_proba
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)
    # For Keras models, use predict
    elif hasattr(model, 'predict'):
        # Need to reshape X_test for the TCN model if it's a Keras model
        if isinstance(X_test, pd.DataFrame):
             X_test_reshaped = X_test.values.reshape(X_test.shape[0], X_test.shape[1], 1)
        else:
            X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        y_score = model.predict(X_test_reshaped)
    else:
        return "Model does not support probability prediction."

    auc_scores = {}
    for i in range(len(label_binarizer.classes_)):
        class_name = label_binarizer.classes_[i]
        try:
            auc = roc_auc_score(y_test_binarized[:, i], y_score[:, i])
            auc_scores[class_name] = auc
        except ValueError as e:
            auc_scores[class_name] = f"Could not calculate AUC: {e}"
    return auc_scores

# Calculate and print AUC for each model
auc_rf = calculate_auc(model, X_test, y_test_binarized, lb)
print("Random Forest:")
for class_name, auc_score in auc_rf.items():
    print(f"  {class_name}: {auc_score:.4f}" if isinstance(auc_score, float) else f"  {class_name}: {auc_score}")
print("-" * 30)

auc_dt = calculate_auc(decision_tree_model, X_test, y_test_binarized, lb)
print("Decision Tree:")
for class_name, auc_score in auc_dt.items():
     print(f"  {class_name}: {auc_score:.4f}" if isinstance(auc_score, float) else f"  {class_name}: {auc_score}")
print("-" * 30)

auc_lr = calculate_auc(log_reg_model, X_test, y_test_binarized, lb)
print("Logistic Regression:")
for class_name, auc_score in auc_lr.items():
     print(f"  {class_name}: {auc_score:.4f}" if isinstance(auc_score, float) else f"  {class_name}: {auc_score}")
print("-" * 30)

auc_nb = calculate_auc(nb_model, X_test, y_test_binarized, lb)
print("Gaussian Naive Bayes:")
for class_name, auc_score in auc_nb.items():
     print(f"  {class_name}: {auc_score:.4f}" if isinstance(auc_score, float) else f"  {class_name}: {auc_score}")
print("-" * 30)

# For TCN with Attention, use the already reshaped X_test_tcn
auc_tcn_attention = calculate_auc(tcn_attention_model, X_test_tcn, y_test_binarized, lb)
print("TCN with Attention:")
for class_name, auc_score in auc_tcn_attention.items():
     print(f"  {class_name}: {auc_score:.4f}" if isinstance(auc_score, float) else f"  {class_name}: {auc_score}")
print("-" * 30)

## 🕸️ Step 16: Radar Plot — Overall Model Comparison
A **Radar Plot** gives a visual 360° comparison of all models across all evaluation metrics.  
This helps identify which model performs best overall at a single glance.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Overall accuracy scores from previous calculations
model_names = ['Random Forest', 'Logistic Regression', 'Gaussian Naive Bayes', 'TCN with Attention']
accuracies = [model.score(X_test, y_test), accuracy_lr, accuracy_nb, accuracy_tcn_attention]

# Create a DataFrame for the radar plot
# We need to reshape the data for a radar plot with a single metric
radar_df_accuracy = pd.DataFrame({
    'Model': model_names,
    'Accuracy': accuracies
})

# --- Create the Radar Plot for overall accuracy ---

labels = np.array(radar_df_accuracy['Model'])
num_vars = len(labels)

# Calculate angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# Complete the loop for plotting
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Plot data for each model's accuracy
values = radar_df_accuracy['Accuracy'].tolist()
values += values[:1] # Complete the loop
ax.plot(angles, values, 'o-', linewidth=2, label='Accuracy')

# Add labels and title
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_title('Overall Model Accuracy Comparison (Radar Plot)', size=15, color='black', y=1.1)
ax.grid(True)
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
ax.set_ylim(0, 1) # Set the limits for accuracy

plt.show()

## ✅ Conclusion

In this project we built and compared **5 machine learning models** for audio-based danger detection of women and children:

| Model | Type |
|-------|------|
| Random Forest | Classical ML |
| Logistic Regression | Classical ML |
| Gaussian Naive Bayes | Classical ML |
| Decision Tree | Classical ML |
| TCN with Attention | Deep Learning |

### Key Takeaways:
- **TCN with Attention** achieved the best performance due to its ability to capture temporal audio patterns
- **Random Forest** was the best performing classical ML model
- Audio features like **MFCCs** proved to be the most informative for classification
- This system can be extended into a **real-time mobile safety application**

---

## 🔮 Future Work
- [ ] Deploy as a real-time mobile application
- [ ] Integrate with GPS for automatic emergency alerts
- [ ] Expand dataset with more diverse audio samples
- [ ] Try Transformer-based models for better accuracy